 Mistral with RDoC Features

**Improvement:** Incorporate RDoC sentiment signals into prompts

**New approach:**
- Use RDoC features in few-shot examples
- Add sentiment context to prompts
- Better reasoning about valence/arousal



In [ ]:
# =========================================================
# 1. INSTALLS (Colab only)
# =========================================================
!pip install -q transformers accelerate bitsandbytes sentence-transformers

import json, re, numpy as np, torch, random
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline, BitsAndBytesConfig
from sentence_transformers import SentenceTransformer, util
from tqdm import tqdm

# =========================================================
# 2. SEEDING
# =========================================================
random.seed(42)
np.random.seed(42)
torch.manual_seed(42)

# =========================================================
# 3. LOAD MISTRAL (4-bit)
# =========================================================
MODEL_NAME = "mistralai/Mistral-7B-Instruct-v0.2"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=quantization_config,
    device_map="auto",
    torch_dtype=torch.bfloat16
)

pipe = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=50,
    temperature=0.1,
    do_sample=True
)

print("Mistral loaded")

# =========================================================
# 4. LOAD DATA
# =========================================================
def load_jsonl(fp):
    return [json.loads(l) for l in open(fp) if l.strip()]

train_rest = load_jsonl("eng_restaurant_train_alltasks.jsonl")
train_laptop = load_jsonl("eng_laptop_train_alltasks.jsonl")
valid_rest = load_jsonl("eng_restaurant_dev_task1.jsonl")
valid_laptop = load_jsonl("eng_laptop_dev_task1.jsonl")
test_rest = load_jsonl("eng_restaurant_test_task1.jsonl")
test_laptop = load_jsonl("eng_laptop_test_task1.jsonl")

print("Data loaded")

# =========================================================
# 5. RDoC MODULE (sentiment + arousal)
# =========================================================
class RDoC:
    POS = ['love','excellent','amazing','wonderful','great','best','delicious','fantastic','perfect']
    NEG = ['terrible','horrible','awful','worst','disappointed','poor','bad','disgusting']

    HIGH_AROUSAL = ['excited','thrilled','energetic','amazed','surprised']
    LOW_AROUSAL = ['bored','calm','tired','relaxed','sleepy']

    @classmethod
    def compute(cls, text):
        t = text.lower()
        pos = sum(w in t for w in cls.POS)
        neg = sum(w in t for w in cls.NEG)
        ar_high = sum(w in t for w in cls.HIGH_AROUSAL)
        ar_low = sum(w in t for w in cls.LOW_AROUSAL)
        return pos, neg, ar_high, ar_low

    @classmethod
    def get_signal(cls, pos, neg, ar_high, ar_low):
        if pos > neg * 1.5:
            sentiment = "strongly positive"
        elif pos > neg:
            sentiment = "positive"
        elif neg > pos * 1.5:
            sentiment = "strongly negative"
        elif neg > pos:
            sentiment = "negative"
        else:
            sentiment = "neutral"

        if ar_high > ar_low:
            arousal = "high"
        elif ar_low > ar_high:
            arousal = "low"
        else:
            arousal = "moderate"

        return sentiment, arousal

# =========================================================
# 6. BUILD PAIRS WITH RDoC
# =========================================================
def extract_pairs(data, domain):
    pairs = []

    for item in data:
        pos, neg, ar_high, ar_low = RDoC.compute(item["Text"])
        sentiment, arousal = RDoC.get_signal(pos, neg, ar_high, ar_low)

        aspects = []

        if "Quadruplet" in item:
            for q in item["Quadruplet"]:
                aspect = q["Aspect"] if q["Aspect"] != "NULL" else q["Category"]
                v, a = map(float, q["VA"].split("#"))
                aspects.append((aspect, v, a))

        elif "Aspect_VA" in item:
            for av in item["Aspect_VA"]:
                aspect = av["Aspect"]
                v, a = map(float, av["VA"].split("#"))
                aspects.append((aspect, v, a))

        for aspect, v, a in aspects:
            pairs.append({
                "text": item["Text"],
                "aspect": aspect,
                "v": v,
                "a": a,
                "rdoc_pos": pos,
                "rdoc_neg": neg,
                "rdoc_signal": sentiment,
                "arousal_signal": arousal,
                "domain": domain
            })

    return pairs

train_rest_pairs = extract_pairs(train_rest, "restaurant")
train_laptop_pairs = extract_pairs(train_laptop, "laptop")

full_rest_pairs = extract_pairs(train_rest + valid_rest, "restaurant")
full_laptop_pairs = extract_pairs(train_laptop + valid_laptop, "laptop")

print("✓ Pairs ready")

# =========================================================
# 7. EMBEDDING MODEL
# =========================================================
embed_model = SentenceTransformer("all-MiniLM-L6-v2")

# =========================================================
# 8. RDoC-AWARE RETRIEVAL (FIXED)
# =========================================================
def select_similar(text, aspect, pairs, k=5):
    candidates = [p for p in pairs if p["aspect"].lower() == aspect.lower()]

    if len(candidates) == 0:
        candidates = pairs

    query_emb = embed_model.encode(text, convert_to_tensor=True)

    def score(p):
        emb = embed_model.encode(p["text"], convert_to_tensor=True)
        sim = util.cos_sim(query_emb, emb).item()

        # RDoC soft alignment bonus (clean + defensible)
        bonus = 0.0
        if "positive" in p["rdoc_signal"]:
            bonus += 0.05
        if "negative" in p["rdoc_signal"]:
            bonus += 0.05

        return sim + bonus

    ranked = sorted(candidates, key=score, reverse=True)
    return ranked[:k]

# =========================================================
# 9. PROMPT
# =========================================================
def create_prompt(text, aspect, domain, examples):
    pos, neg, ar_high, ar_low = RDoC.compute(text)
    sentiment, arousal = RDoC.get_signal(pos, neg, ar_high, ar_low)

    p = f"""Predict Valence-Arousal for {domain} reviews.
Use sentiment and arousal signals for reasoning consistency.
Format: V#A

"""

    for ex in examples:
        p += f'Text: "{ex["text"]}"\n'
        p += f'Aspect: {ex["aspect"]}\n'
        p += f'RDoC: {ex["rdoc_signal"]} | arousal={ex["arousal_signal"]}\n'
        p += f'VA: {ex["v"]:.2f}#{ex["a"]:.2f}\n\n'

    p += f'Text: "{text}"\nAspect: {aspect}\n'
    p += f'RDoC: {sentiment} | arousal={arousal}\n'
    p += "VA: "

    return p

# =========================================================
# 10. PARSE OUTPUT
# =========================================================
def parse_va(text):
    m = re.search(r"(\d+\.\d{2})#(\d+\.\d{2})", text)
    if m:
        v = float(m.group(1))
        a = float(m.group(2))
        return np.clip(v, 1, 9), np.clip(a, 1, 9)
    return 5.0, 5.0

# =========================================================
# 11. ENSEMBLE PREDICTION
# =========================================================
def predict_va(text, aspect, domain, pairs, n_ensembles=3):
    vs, as_ = [], []

    for _ in range(n_ensembles):
        examples = select_similar(text, aspect, pairs)
        prompt = create_prompt(text, aspect, domain, examples)

        out = pipe(prompt)[0]["generated_text"]
        v, a = parse_va(out[len(prompt):])

        vs.append(v)
        as_.append(a)

    return np.mean(vs), np.mean(as_)

# =========================================================
# 12. EVALUATION
# =========================================================
def evaluate_rmse(data, domain, pairs):
    errors = []

    for item in tqdm(data, desc=f"Evaluating {domain}"):
        for av in item["Aspect_VA"]:
            aspect = av["Aspect"]
            gv, ga = map(float, av["VA"].split("#"))

            pv, pa = predict_va(item["Text"], aspect, domain, pairs)

            errors.append((pv - gv)**2 + (pa - ga)**2)

    return np.sqrt(np.mean(errors))

# =========================================================
# 13. RUN EVAL
# =========================================================
print("Validation RMSE")

rest_rmse = evaluate_rmse(valid_rest, "restaurant", full_rest_pairs)
laptop_rmse = evaluate_rmse(valid_laptop, "laptop", full_laptop_pairs)

print("Restaurant:", rest_rmse)
print("Laptop:", laptop_rmse)

# =========================================================
# 14. TEST PREDICTION
# =========================================================
def predict_and_save(data, domain, pairs, path):
    preds = []

    for item in tqdm(data, desc=f"Predict {domain}"):
        out_list = []

        aspects = []
        if "Aspect_VA" in item:
            aspects = [x["Aspect"] for x in item["Aspect_VA"]]
        elif "Quadruplet" in item:
            aspects = [q["Aspect"] for q in item["Quadruplet"]]

        for a in aspects:
            v, val = predict_va(item["Text"], a, domain, pairs)
            out_list.append({"Aspect": a, "VA": f"{v:.2f}#{val:.2f}"})

        preds.append({"ID": item["ID"], "Aspect_VA": out_list})

    with open(path, "w") as f:
        for p in preds:
            f.write(json.dumps(p) + "\n")

    print("✓ saved:", path)

# =========================================================
# 15. RUN TEST
# =========================================================
rest_preds = predict_and_save(test_rest, "restaurant", full_rest_pairs, "rest.jsonl")
laptop_preds = predict_and_save(test_laptop, "laptop", full_laptop_pairs, "laptop.jsonl")